# Step 3: Data Cleaning & Preprocessing
**Project:** Taobao CTR Prediction — Big Data Pipeline  
**Target:** `clk` (1 = clicked, 0 = not clicked)  
**Steps:** Missing Values → Duplicates → Join Tables → Feature Engineering → Encoding → Scaling → Save

## Cell 0 — HADOOP Setup 

In [13]:
import os

os.environ["HADOOP_HOME"]     = "D:\\Apps\\hadoop"
os.environ["hadoop.home.dir"] = "D:\\Apps\\hadoop"
os.environ["PATH"]            = "D:\\Apps\\hadoop\\bin;" + os.environ["PATH"]

print(" HADOOP_HOME:", os.environ["HADOOP_HOME"])
print(" Hadoop configured!")

 HADOOP_HOME: D:\Apps\hadoop
 Hadoop configured!


## Cell 1 — Spark Session

In [14]:
import findspark
findspark.init()

import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnull, count, mean, hour,
    from_unixtime, dayofweek, sum as spark_sum,
    round as spark_round
)
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
import pandas as pd

DATA_PATH   = r"D:\Apps\ProjectBigData\archive"
OUTPUT_PATH = "D:/Apps/ProjectBigData/"

os.makedirs(OUTPUT_PATH + "processed_data", exist_ok=True)

spark = SparkSession.builder \
    .appName("Taobao_CTR_Preprocessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Version:", spark.version)
print(" Spark Session Started!")

 Spark Version: 4.1.1
 Spark Session Started!


## Cell 2 — Load Data

In [15]:
print(" Loading datasets...\n")

raw_sample_df = spark.read.csv(
    os.path.join(DATA_PATH, "raw_sample.csv"),
    header=True, inferSchema=True
)
ad_feature_df = spark.read.csv(
    os.path.join(DATA_PATH, "ad_feature.csv"),
    header=True, inferSchema=True
)
user_profile_df = spark.read.csv(
    os.path.join(DATA_PATH, "user_profile.csv"),
    header=True, inferSchema=True
)

print(f" raw_sample   : {raw_sample_df.count():,} records")
print(f" ad_feature   : {ad_feature_df.count():,} records")
print(f" user_profile : {user_profile_df.count():,} records")

 Loading datasets...

 raw_sample   : 26,557,961 records
 ad_feature   : 846,811 records
 user_profile : 1,061,768 records


## Cell 3 — Handle Missing Values

In [16]:
print(" STEP 1: HANDLING MISSING VALUES")
print("=" * 50)

ad_feature_clean = ad_feature_df.fillna({"brand": "0"})


user_profile_clean = user_profile_df.fillna({
    "pvalue_level"          : 0,
    "new_user_class_level " : 0,
    "cms_segid"             : 0
})

def check_nulls(df, name):
    null_expr = [spark_sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in df.columns]
    result    = df.select(null_expr).collect()[0].asDict()
    total     = sum(result.values())
    print(f"  {name}: {total} nulls remaining")

check_nulls(ad_feature_clean,   "ad_feature")
check_nulls(user_profile_clean, "user_profile")
print("\n Missing values handled!")

 STEP 1: HANDLING MISSING VALUES
  ad_feature: 0 nulls remaining
  user_profile: 0 nulls remaining

 Missing values handled!


## Cell 4 — Remove Duplicates

In [17]:
print(" STEP 2: REMOVING DUPLICATES")
print("=" * 50)

before           = raw_sample_df.count()
raw_sample_clean = raw_sample_df.dropDuplicates(["user", "time_stamp", "adgroup_id"])
after            = raw_sample_clean.count()

print(f"  raw_sample Before : {before:,}")
print(f"  raw_sample After  : {after:,}")
print(f"  Duplicates Removed: {before - after:,}")

ad_feature_clean = ad_feature_clean.dropDuplicates(["adgroup_id"])
print(f"\n  ad_feature unique ads: {ad_feature_clean.count():,}")
print("\n Duplicates removed!")

 STEP 2: REMOVING DUPLICATES
  raw_sample Before : 26,557,961
  raw_sample After  : 26,557,961
  Duplicates Removed: 0

  ad_feature unique ads: 846,811

 Duplicates removed!


## Cell 5 — Join Tables

In [18]:
print(" STEP 3: JOINING TABLES")
print("=" * 50)

df = raw_sample_clean.join(ad_feature_clean, on="adgroup_id", how="left")
print(f"  After join with ad_feature   : {df.count():,} records")

df = df.join(
    user_profile_clean,
    df["user"] == user_profile_clean["userid"],
    how="left"
).drop("userid")

print(f"  After join with user_profile : {df.count():,} records")
print(f"  Total Columns                : {len(df.columns)}")
print("\n All tables joined!")

 STEP 3: JOINING TABLES
  After join with ad_feature   : 26,557,961 records
  After join with user_profile : 26,557,961 records
  Total Columns                : 19

 All tables joined!


## Cell 6 — Feature Engineering

In [19]:
print(" STEP 4: FEATURE ENGINEERING")
print("=" * 50)

df = df.withColumn("hour",        hour(from_unixtime(col("time_stamp"))))
df = df.withColumn("day_of_week", dayofweek(from_unixtime(col("time_stamp"))))
df = df.withColumn("is_weekend",  when(col("day_of_week").isin([1, 7]), 1).otherwise(0))
df = df.withColumn("time_segment",
    when(col("hour").between(0, 6),   0)
   .when(col("hour").between(7, 11),  1)
   .when(col("hour").between(12, 17), 2)
   .otherwise(3)
)

ad_ctr = df.groupBy("adgroup_id").agg(
    spark_round(mean("clk"), 4).alias("ad_historical_ctr"),
    count("*").alias("ad_impression_count")
)
df = df.join(ad_ctr, on="adgroup_id", how="left")

user_activity = df.groupBy("user").agg(
    spark_sum("clk").alias("user_total_clicks"),
    count("*").alias("user_total_impressions")
)
user_activity = user_activity.withColumn(
    "user_ctr",
    spark_round(col("user_total_clicks") / col("user_total_impressions"), 4)
)
df = df.join(user_activity, on="user", how="left")

df = df.fillna({
    "ad_historical_ctr"     : 0.0,
    "ad_impression_count"   : 0,
    "user_total_clicks"     : 0,
    "user_total_impressions": 0,
    "user_ctr"              : 0.0,
    "price"                 : 0.0,
    "cms_segid"             : 0,
    "cms_group_id"          : 0,
    "final_gender_code"     : 0,
    "age_level"             : 0,
    "pvalue_level"          : 0,
    "shopping_level"        : 0,
    "occupation"            : 0
})

print("   hour, day_of_week, is_weekend, time_segment")
print("   ad_historical_ctr, ad_impression_count")
print("   user_total_clicks, user_ctr")
print(f"\n  Total Columns Now: {len(df.columns)}")
print("\n Feature engineering done!")

 STEP 4: FEATURE ENGINEERING
   hour, day_of_week, is_weekend, time_segment
   ad_historical_ctr, ad_impression_count
   user_total_clicks, user_ctr

  Total Columns Now: 28

 Feature engineering done!


## Cell 7 — Encoding + Assembly + Scaling

In [ ]:
print(" STEP 5: ENCODING & FEATURE ASSEMBLY")
print("=" * 50)

# Encode all string columns automatically
string_cols  = [c for c, t in df.dtypes if t == "string"]
encoded_map  = {}

for s_col in string_cols:
    out_col = s_col + "_encoded"
    if out_col not in df.columns:
        indexer = StringIndexer(inputCol=s_col, outputCol=out_col, handleInvalid="keep")
        df      = indexer.fit(df).transform(df)
        print(f"   {s_col} → {out_col}")
    encoded_map[s_col] = out_col

# Drop old vectors if exist
for drop_col in ["features_raw", "features"]:
    if drop_col in df.columns:
        df = df.drop(drop_col)

# Feature columns
base_feature_cols = [
    "adgroup_id", "cate_id", "campaign_id", "brand", "price",
    "pid", "ad_historical_ctr", "ad_impression_count",
    "cms_segid", "cms_group_id", "final_gender_code",
    "age_level", "pvalue_level", "shopping_level",
    "occupation", "user_total_clicks", "user_ctr",
    "hour", "day_of_week", "is_weekend", "time_segment"
]

feature_cols = [
    encoded_map.get(c, c) for c in base_feature_cols
    if encoded_map.get(c, c) in df.columns
]

print(f"\n  Features ({len(feature_cols)}): {feature_cols}")

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"
)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True, withStd=True
)

pipeline       = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df)
df_final       = pipeline_model.transform(df)

print("\n   VectorAssembler → features_raw")
print("   StandardScaler  → features")
print(f"\n  df_final rows: {df_final.count():,}")
print("\n Encoding & Assembly complete!")

 STEP 5: ENCODING & FEATURE ASSEMBLY
   pid → pid_encoded
   brand → brand_encoded

  Features (21): ['adgroup_id', 'cate_id', 'campaign_id', 'brand_encoded', 'price', 'pid_encoded', 'ad_historical_ctr', 'ad_impression_count', 'cms_segid', 'cms_group_id', 'final_gender_code', 'age_level', 'pvalue_level', 'shopping_level', 'occupation', 'user_total_clicks', 'user_ctr', 'hour', 'day_of_week', 'is_weekend', 'time_segment']


## Cell 8 — Class Imbalance

In [9]:
print(" STEP 6: HANDLE CLASS IMBALANCE")
print("=" * 50)

total     = df_final.count()
clicks    = df_final.filter(col("clk") == 1).count()
no_clicks = total - clicks

weight_for_click    = total / (2 * clicks)
weight_for_no_click = total / (2 * no_clicks)

df_final = df_final.withColumn(
    "class_weight",
    when(col("clk") == 1, weight_for_click).otherwise(weight_for_no_click)
)

print(f"  Clicks    : {clicks:,}  ({clicks/total*100:.2f}%)")
print(f"  No Click  : {no_clicks:,} ({no_clicks/total*100:.2f}%)")
print(f"  Weight(1) : {weight_for_click:.2f}")
print(f"  Weight(0) : {weight_for_no_click:.2f}")
print("\n Class weights added!")

 STEP 6: HANDLE CLASS IMBALANCE
  Clicks    : 1,366,056  (5.14%)
  No Click  : 25,191,905 (94.86%)
  Weight(1) : 9.72
  Weight(0) : 0.53

 Class weights added!


## Cell 9 — Train/Test Split

In [10]:
print(" STEP 7: TRAIN / TEST SPLIT")
print("=" * 50)

TEST_START_TIMESTAMP = 1494604800

train_df = df_final.filter(col("time_stamp") <  TEST_START_TIMESTAMP)
test_df  = df_final.filter(col("time_stamp") >= TEST_START_TIMESTAMP)

train_count = train_df.count()
test_count  = test_df.count()
total_count = train_count + test_count

print(f"  Train (days 1-7) : {train_count:,} ({train_count/total_count*100:.1f}%)")
print(f"  Test  (day 8)    : {test_count:,}  ({test_count/total_count*100:.1f}%)")
print("\n Train/Test split done!")

 STEP 7: TRAIN / TEST SPLIT
  Train (days 1-7) : 23,249,296 (87.5%)
  Test  (day 8)    : 3,308,665  (12.5%)

 Train/Test split done!


## Cell 10 — Save Data

In [11]:
print(" STEP 8: SAVE PROCESSED DATA")
print("=" * 50)

final_cols    = ["features", "clk", "class_weight"]
TRAIN_PATH    = OUTPUT_PATH + "processed_data/train"
TEST_PATH     = OUTPUT_PATH + "processed_data/test"
PIPELINE_PATH = OUTPUT_PATH + "pipeline_model"

print(" Saving train data...")
train_df.select(final_cols).write.mode("overwrite").parquet(TRAIN_PATH)
print(f"   Train saved → {TRAIN_PATH}")

print(" Saving test data...")
test_df.select(final_cols).write.mode("overwrite").parquet(TEST_PATH)
print(f"   Test  saved → {TEST_PATH}")

print(" Saving pipeline...")
pipeline_model.write().overwrite().save(PIPELINE_PATH)
print(f"   Pipeline saved → {PIPELINE_PATH}")

print("\n All data saved!")

 STEP 8: SAVE PROCESSED DATA
 Saving train data...
   Train saved → D:/Apps/ProjectBigData/processed_data/train
 Saving test data...
   Test  saved → D:/Apps/ProjectBigData/processed_data/test
 Saving pipeline...
   Pipeline saved → D:/Apps/ProjectBigData/pipeline_model

 All data saved!


## Cell 11 — Final Summary

In [12]:
print("=" * 60)
print(" PREPROCESSING COMPLETE SUMMARY")
print("=" * 60)

train_check = spark.read.parquet(TRAIN_PATH)
test_check  = spark.read.parquet(TEST_PATH)

print(f"""
   Steps Completed:
     1. Missing values    → filled with 0
     2. Duplicates        → removed
     3. Tables joined     → raw + ad_feature + user_profile
     4. Feature Engineering:
        - hour, day_of_week, is_weekend, time_segment
        - ad_historical_ctr, ad_impression_count
        - user_total_clicks, user_ctr
     5. Encoding          → StringIndexer
     6. Scaling           → StandardScaler
     7. Class imbalance   → class_weight added
     8. Train/Test split  → time-based

   Saved Files:
     Train    → {TRAIN_PATH}
     Test     → {TEST_PATH}
     Pipeline → {PIPELINE_PATH}

   Final Dataset:
     Train : {train_check.count():,} records
     Test  : {test_check.count():,} records
     Target: clk (0/1)

    
""")

spark.stop()
print(" 02_Preprocessing.ipynb — Complete!")

 PREPROCESSING COMPLETE SUMMARY

   Steps Completed:
     1. Missing values    → filled with 0
     2. Duplicates        → removed
     3. Tables joined     → raw + ad_feature + user_profile
     4. Feature Engineering:
        - hour, day_of_week, is_weekend, time_segment
        - ad_historical_ctr, ad_impression_count
        - user_total_clicks, user_ctr
     5. Encoding          → StringIndexer
     6. Scaling           → StandardScaler
     7. Class imbalance   → class_weight added
     8. Train/Test split  → time-based

   Saved Files:
     Train    → D:/Apps/ProjectBigData/processed_data/train
     Test     → D:/Apps/ProjectBigData/processed_data/test
     Pipeline → D:/Apps/ProjectBigData/pipeline_model

   Final Dataset:
     Train : 23,249,296 records
     Test  : 3,308,665 records
     Target: clk (0/1)



 02_Preprocessing.ipynb — Complete!
